In [2]:
import pandas as pd

# Load dataset
df = pd.read_csv(r"E:\AAA_projects\Cab Travel\travel_raw.csv")

# -----------------------------------
# 1. Clean column names
# -----------------------------------
df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')

# check column names
print(df.columns)

Index(['start_date', 'end_date', 'category', 'start', 'stop', 'miles',
       'purpose'],
      dtype='object')


In [3]:
# -----------------------------------
# 2. Check missing values
# -----------------------------------
print(df.isnull().sum())

start_date      0
end_date        1
category        1
start           1
stop            1
miles           0
purpose       503
dtype: int64


In [4]:
# fill text columns with mode
df['category'] = df['category'].fillna(df['category'].mode()[0])
df['start'] = df['start'].fillna(df['start'].mode()[0])
df['stop'] = df['stop'].fillna(df['stop'].mode()[0])

# fill purpose with mode
df['purpose'] = df['purpose'].fillna(df['purpose'].mode()[0])

# fill missing end_date using forward fill
df['end_date'] = df['end_date'].fillna(method='ffill')

# check again
print(df.isnull().sum())
                                     

start_date    0
end_date      0
category      0
start         0
stop          0
miles         0
purpose       0
dtype: int64


C:\Users\ahama\AppData\Local\Temp\ipykernel_8784\1999413379.py:10: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['end_date'] = df['end_date'].fillna(method='ffill')


In [5]:
# Forward fill was used because the missing date value was very limited 
#and date continuity was maintained using the nearest previous valid timestamp.

In [6]:
# -----------------------------------
#  Check duplicates
# -----------------------------------
print("Duplicate rows:")
print(df.duplicated().sum())

# View duplicate rows
print(df[df.duplicated()])

# Remove duplicates
df = df.drop_duplicates()

Duplicate rows:
1
          start_date         end_date  category   start  stop  miles  purpose
492  6/28/2016 23:34  6/28/2016 23:59  Business  Durham  Cary    9.9  Meeting


In [7]:
# -----------------------------------
# 2. Check datatypes
# -----------------------------------
print("Datatypes before fixing:")
print(df.dtypes)

Datatypes before fixing:
start_date     object
end_date       object
category       object
start          object
stop           object
miles         float64
purpose        object
dtype: object


In [13]:
# -----------------------------------
# 4. Create new feature columns
# -----------------------------------

# trip duration in minutes
df['trip_duration'] = (df['end_date'] - df['start_date']).dt.total_seconds() / 60

# extract hour
df['hour'] = df['start_date'].dt.hour

# extract day name
df['day'] = df['start_date'].dt.day_name()

# extract month
df['month'] = df['start_date'].dt.month_name()

# weekend column
df['is_weekend'] = df['day'].isin(['Saturday', 'Sunday'])

# route column
df['route'] = df['start'] + ' - ' + df['stop']

# high distance trip
df['high_distance'] = df['miles'] > 20

In [14]:
# -----------------------------------
#  Fix correct datatypes
# -----------------------------------

# datetime columns
df['start_date'] = pd.to_datetime(df['start_date'])
df['end_date'] = pd.to_datetime(df['end_date'])

# text columns
df['category'] = df['category'].astype(str)
df['start'] = df['start'].astype(str)
df['stop'] = df['stop'].astype(str)
df['purpose'] = df['purpose'].astype(str)
df['day'] = df['day'].astype(str)
df['month'] = df['month'].astype(str)
df['route'] = df['route'].astype(str)

# numeric columns
df['miles'] = pd.to_numeric(df['miles'])
df['trip_duration'] = pd.to_numeric(df['trip_duration'])
df['hour'] = pd.to_numeric(df['hour'])

# boolean column
df['is_weekend'] = df['is_weekend'].astype(bool)
df['high_distance'] = df['high_distance'].astype(bool)


In [15]:
# -----------------------------------
# 5. Basic Analysis
# -----------------------------------

# top starting locations
print("Top Start Locations")
print(df['start'].value_counts().head(10))

# top stop locations
print("Top Stop Locations")
print(df['stop'].value_counts().head(10))

# top routes
print("Top Routes")
print(df['route'].value_counts().head(10))

# category count
print("Category Count")
print(df['category'].value_counts())

# purpose count
print("Purpose Count")
print(df['purpose'].value_counts())

# average miles
print("Average Miles")
print(df['miles'].mean())

# longest trips
print("Longest Trips")
print(df[['start','stop','miles']].sort_values(by='miles', ascending=False).head(10))

# hourly trips
print("Trips by Hour")
print(df['hour'].value_counts().sort_index())

# day wise trips
print("Trips by Day")
print(df['day'].value_counts())

Top Start Locations
start
Cary                202
Unknown Location    148
Morrisville          85
Whitebridge          68
Islamabad            57
Durham               36
Lahore               36
Raleigh              28
Kar?chi              27
Westpark Place       17
Name: count, dtype: int64
Top Stop Locations
stop
Cary                203
Unknown Location    149
Morrisville          84
Whitebridge          65
Islamabad            58
Durham               36
Lahore               36
Raleigh              29
Kar?chi              26
Apex                 17
Name: count, dtype: int64
Top Routes
route
Unknown Location - Unknown Location    86
Morrisville - Cary                     75
Cary - Morrisville                     67
Cary - Cary                            54
Cary - Durham                          36
Durham - Cary                          31
Unknown Location - Islamabad           28
Islamabad - Unknown Location           28
Lahore - Lahore                        27
Islamabad - Islamabad  

In [16]:
df.to_csv(r"E:\AAA_projects\Cab Travel\travel_final_cleaned.csv", index=False)

print("File saved successfully")

File saved successfully
